# Equity Duration Notebook

This notebook constructs several **duration** measures for STOXX Europe 600 constituents:

1. **Operating cash-flow timing duration** (undiscounted)
2. **Discounted cash-flow duration** with fixed discount rate \(r = 12.5\%\)
3. **Discounted cash-flow duration** with firm-specific **CAPM cost of equity**
4. **Empirical (interest-rate sensitivity) duration** using daily changes in **2Y EUR OIS** (RIC `EUREON2Y=`)

> **Note on cash flows:** `CFPSMean_*` in this project is *cash flow from operations per share* (CFO per share), not net payouts to equity holders.


## 0. Setup

- Requires: `lseg.data` (LSEG Data Library), `pandas`, `numpy`
- You may need to authenticate / configure LSEG before running.


In [ ]:
import re
import numpy as np
import pandas as pd
import lseg.data as ld
from pathlib import Path
import time
import random

#Speicherstruktur für Intermediate und Final Output
BASE_DIR = Path(
    "/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data"
)
BASE_DIR.mkdir(parents=True, exist_ok=True)

(TABLE_DIR := BASE_DIR / "tables").mkdir(exist_ok=True)
(DATA_DIR  := BASE_DIR / "intermediate").mkdir(exist_ok=True)

def save_parquet(df: pd.DataFrame, name: str):
    path = DATA_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved: {path}")

def load_parquet(name: str) -> pd.DataFrame:
    path = DATA_DIR / f"{name}.parquet"
    return pd.read_parquet(path)


## 1. Load the STOXX 600 (EURO subset) constituent list

Expected columns (names can vary):
- RIC
- Company name
- Country of headquarters
- Market cap

Adjust `INPUT_XLSX` if needed.


In [2]:
# --- User input ---
INPUT_XLSX = TABLE_DIR / "stoxx600_membership_matrix_1999_2025_eurohq.xlsx"
SHEET_NAME = 0  # or a sheet name string

df_const = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_NAME)

# Robust column detection
ric_col = next(c for c in df_const.columns if "RIC" in c.upper())
name_col = next(c for c in df_const.columns if "COMPANY" in c.upper() or "NAME" in c.upper())
country_col = next(c for c in df_const.columns if "COUNTRY" in c.upper())

df_basic = df_const[[ric_col, name_col, country_col]].copy()
df_basic.columns = ["RIC", "CompanyName", "CountryHQ"]

df_basic = (
    df_basic
    .dropna(subset=["RIC"])
    .drop_duplicates(subset=["RIC"])
    .sort_values("RIC")
    .reset_index(drop=True)
)

print(df_basic.head())
print("Number of firms:", len(df_basic))


            RIC                                        CompanyName  \
0  1COVG.DE^L25                                        Covestro AG   
1        1U1.DE                                             1&1 AG   
2         A2.MI                                            A2A SpA   
3        A3M.MC  Atresmedia Corporacion de Medios de Comunicaci...   
4       AALB.AS                                        Aalberts NV   

     CountryHQ  
0      Germany  
1      Germany  
2        Italy  
3        Spain  
4  Netherlands  
Number of firms: 667


## 2. Helper functions (LSEG pulls + finance math)

We define small helper functions for:
- Year-end price
- ROE (FY0)
- 5-year beta
- CAPM cost of equity
- Duration measures (discounted + undiscounted)


In [3]:
def get_year_end_price(ric: str, year: int = 2025, field: str = "TR.PriceClose") -> float:
    """Get the last available daily price in a given calendar year."""
    hist = ld.get_history(
        universe=[ric],
        fields=[field],
        start=f"{year}-01-01",
        end=f"{year}-12-31",
        interval="daily"
    )
    if hist is None or hist.empty:
        return np.nan

    # hist typically has Date as index and one column for the field
    df = hist.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()
        date_col = next(c for c in df.columns if "date" in str(c).lower())
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.set_index(date_col)

    # take last non-missing value
    s = pd.to_numeric(df.iloc[:, 0], errors="coerce").dropna()
    return float(s.iloc[-1]) if len(s) else np.nan


def get_roe_fy0(ric: str) -> float:
    """ROE (FY0) from LSEG. Returns decimal (0.15 = 15%)."""
    roe_df = ld.get_data(
        universe=[ric],
        fields=["TR.F.ReturnAvgTotEqPct(period=FY0)"]
    )
    if roe_df is None or roe_df.empty:
        return np.nan

    roe_col = next((c for c in roe_df.columns if "Return on Average Total Equity" in c), None)
    if roe_col is None:
        # fallback: take the only non-id column
        roe_col = roe_df.columns[-1]

    roe = roe_df[roe_col].iloc[0]
    if roe is None:
        return np.nan

    return float(roe) / 100.0


def get_beta_5y(ric: str) -> float:
    """5Y beta from LSEG."""
    beta_df = ld.get_data(universe=[ric], fields=["TR.BetaFiveYear"])
    if beta_df is None or beta_df.empty:
        return np.nan

    beta_col = next((c for c in beta_df.columns if "Beta" in c), None)
    if beta_col is None:
        beta_col = beta_df.columns[-1]

    b = beta_df[beta_col].iloc[0]
    return float(b) if b is not None else np.nan


def coe_capm(beta: float, rf: float, erp: float) -> float:
    """CAPM cost of equity in decimals."""
    if beta is None or (isinstance(beta, float) and np.isnan(beta)):
        return np.nan
    return rf + beta * erp


def duration_discounted(CF: np.ndarray, P: float, r: float) -> float:
    """Implied duration with terminal value inferred from price."""
    if not np.isfinite(P) or P <= 0 or not np.isfinite(r) or r <= 0:
        return np.nan
    CF = np.asarray(CF, dtype=float)
    CF = CF[np.isfinite(CF)]
    T = len(CF)
    if T < 2:
        return np.nan

    t = np.arange(1, T + 1, dtype=float)
    pv_cf = CF / (1.0 + r) ** t
    pv_sum = pv_cf.sum()

    term1 = (t * pv_cf).sum() / P
    term2 = (T + (1.0 + r) / r) * ((P - pv_sum) / P)
    return float(term1 + term2)


def duration_undiscounted(CF: np.ndarray) -> float:
    """Cash-flow timing duration (undiscounted)."""
    CF = np.asarray(CF, dtype=float)
    CF = CF[np.isfinite(CF)]
    T = len(CF)
    if T < 2:
        return np.nan
    t = np.arange(1, T + 1, dtype=float)
    return float((t * CF).sum() / CF.sum()) if CF.sum() != 0 else np.nan


## 3. Pull firm-level inputs (price, ROE, beta) and compute CAPM \(r_i\)

- **Price (Dec 31, 2025):** last available daily close in 2025
- **ROE FY0:** used for descriptive stats (not for duration construction here)
- **Beta (5Y):** used for CAPM \(r_i\)


In [5]:
# --- User choices for CAPM ---
RF = 0.025   # 2.5% risk-free (decimal)
ERP = 0.05   # 5% equity risk premium (decimal)

ld.open_session()

# Year-end price
prices = []
for ric in df_basic["RIC"]:
    try:
        prices.append(get_year_end_price(ric, year=2025, field="TR.PriceClose"))
    except Exception:
        prices.append(np.nan)
df_basic["Price_2025_12_31"] = prices

# ROE FY0
roe_vals = []
for ric in df_basic["RIC"]:
    try:
        roe_vals.append(get_roe_fy0(ric))
    except Exception:
        roe_vals.append(np.nan)
df_basic["ROE_FY0"] = roe_vals

# Beta + CAPM CoE
betas = []
coes = []
for ric in df_basic["RIC"]:
    try:
        b = get_beta_5y(ric)
    except Exception:
        b = np.nan
    betas.append(b)
    coes.append(coe_capm(b, rf=RF, erp=ERP))

df_basic["BETA_5Y"] = betas
df_basic["COE_CAPM"] = coes

ld.close_session()

df_basic[["RIC","Price_2025_12_31","ROE_FY0","BETA_5Y","COE_CAPM"]].head()


,RIC,Price_2025_12_31,ROE_FY0,BETA_5Y,COE_CAPM
0,1COVG.DE^L25,59.98,-0.040911,0.977601,0.073880
1,1U1.DE,24.75,0.035517,0.259409,0.037970
2,A2.MI,2.31,0.165109,0.869877,0.068494
3,A3M.MC,4.88,0.151072,0.921566,0.071078
4,AALB.AS,28.06,0.073169,1.345772,0.092289


In [6]:
# Vorbereitung zum Speichern als Parquet

def make_arrow_safe(df: pd.DataFrame) -> pd.DataFrame:
    df2 = df.copy()

    for c in df2.columns:
        # datetime ok
        if pd.api.types.is_datetime64_any_dtype(df2[c]):
            continue

        # Pandas nullable integers/booleans -> float/bool/object
        if str(df2[c].dtype) in ["Int64", "Int32", "Int16", "Int8"]:
            df2[c] = df2[c].astype("float64")
        elif str(df2[c].dtype) in ["boolean"]:
            df2[c] = df2[c].astype("object")

        # pandas string dtype -> object (safe)
        elif pd.api.types.is_string_dtype(df2[c]) and str(df2[c].dtype).startswith("string"):
            df2[c] = df2[c].astype("object")

        # mixed objects: keep, but ensure pandas.NA -> np.nan
        if df2[c].dtype == "object":
            df2[c] = df2[c].where(df2[c].notna(), None)

    return df2

def save_parquet(df: pd.DataFrame, name: str):
    path = DATA_DIR / f"{name}.parquet"
    df_safe = make_arrow_safe(df)
    df_safe.to_parquet(path, index=False, engine="pyarrow")
    print(f"Saved: {path}")

# Tabelle in intermediate Speichern

firm_cols = [
    "RIC", "CompanyName", "CountryHQ", "Price_2025_12_31",
    "ROE_FY0", "BETA_5Y", "COE_CAPM"
]

df_firm_inputs = df_basic[firm_cols].copy()

save_parquet(df_firm_inputs, "firm_inputs")

Saved: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/firm_inputs.parquet


## 4. Analyst operating cash-flow forecasts (CFPS)

This notebook expects these columns already in `df_basic`:

- `CFPSMean_FY25` … `CFPSMean_FY29`


In [ ]:
years = [2025, 2026, 2027, 2028, 2029]
rics = df_basic["RIC"].dropna().astype(str).unique().tolist()

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

ld.open_session()

try:
    for y in years:
        col_name = f"CFPSMean_FY{str(y)[-2:]}"
        if col_name not in df_basic.columns:
            df_basic[col_name] = np.nan

        # Sammle alle Batch-Ergebnisse in einem Mapping (RIC -> value)
        mapping = {}

        for batch in chunks(rics, 200):
            df_y = ld.get_data(
                universe=batch,
                fields=["TR.CFPSMean"],
                parameters={"SDate": f"{y}-12-01", "EDate": f"{y}-12-01", "FRQ": "FY"}
            )

            if df_y is None or df_y.empty:
                continue

            # Stabilisiert dtypes (hilft gegen LSEG/pandas downcasting warnings)
            df_y = df_y.infer_objects(copy=False)

            # Identifier-Spalte robust finden (manchmal "Instrument")
            id_col = "Instrument" if "Instrument" in df_y.columns else df_y.columns[0]

            # CFPS-Spalte robust finden
            cfps_col = next(
                (c for c in df_y.columns
                 if ("Cash Flow Per Share" in str(c)) and ("Mean" in str(c))),
                None
            )
            if cfps_col is None:
                # Fallback: wenn LSEG dir direkt "TR.CFPSMean" als Spaltennamen gibt
                cfps_col = next((c for c in df_y.columns if "CFPS" in str(c).upper()), None)

            if cfps_col is None:
                continue

            tmp = df_y[[id_col, cfps_col]].copy()
            tmp = tmp.rename(columns={id_col: "RIC", cfps_col: col_name})
            tmp["RIC"] = tmp["RIC"].astype(str)
            tmp[col_name] = pd.to_numeric(tmp[col_name], errors="coerce")

            # Update mapping (letzter gültiger Wert gewinnt)
            tmp = tmp.dropna(subset=[col_name])
            mapping.update(dict(zip(tmp["RIC"], tmp[col_name])))

        # In df_basic schreiben (ohne merges im Loop)
        s_new = df_basic["RIC"].map(mapping)

        # falls schon Werte da sind: keep existing, fill missing with new
        df_basic[col_name] = df_basic[col_name].combine_first(s_new)

finally:
    ld.close_session()

print(df_basic[["RIC", "CompanyName", "CFPSMean_FY25", "CFPSMean_FY26", "CFPSMean_FY27", "CFPSMean_FY28", "CFPSMean_FY29"]].head())
print("Missing FY25:", df_basic["CFPSMean_FY25"].isna().sum())

In [ ]:
cf_cols = ["CFPSMean_FY25", "CFPSMean_FY26", "CFPSMean_FY27", "CFPSMean_FY28", "CFPSMean_FY29"]

# Ensure numeric
for c in cf_cols:
    df_basic[c] = pd.to_numeric(df_basic[c], errors="coerce")

df_basic[["RIC"] + cf_cols].head()

cf_cols = ["CFPSMean_FY25", "CFPSMean_FY26", "CFPSMean_FY27", "CFPSMean_FY28", "CFPSMean_FY29"]

df_cfps = df_basic[["RIC"] + cf_cols].copy()
save_parquet(df_cfps, "cfps_forecasts")


## 5. Duration measures from CFPS

We compute three timing measures:

- `Duration_r125`: discounted duration with fixed \(r=12.5\%\)
- `Duration_CAPM`: discounted duration with firm-specific \(r_i\) from CAPM
- `Duration_undiscounted`: pure timing (no discounting)

All durations use **price per share** and **CFPS** (per share), so units are consistent.


In [ ]:
R_FIXED = 0.125

# Build CF arrays per row
def row_cf_array(row) -> np.ndarray:
    s = pd.to_numeric(pd.Series([row.get(c) for c in cf_cols]), errors="coerce")
    return s.to_numpy(dtype=float)

df_basic["Duration_r125"] = df_basic.apply(
    lambda row: duration_discounted(row_cf_array(row), P=row["Price_2025_12_31"], r=R_FIXED),
    axis=1
)

df_basic["Duration_CAPM"] = df_basic.apply(
    lambda row: duration_discounted(row_cf_array(row), P=row["Price_2025_12_31"], r=row["COE_CAPM"]),
    axis=1
)

df_basic["Duration_undiscounted"] = df_basic.apply(
    lambda row: duration_undiscounted(row_cf_array(row)),
    axis=1
)

df_basic[["RIC","Duration_r125","Duration_CAPM","Duration_undiscounted"]].head()


## 6. Empirical duration: interest-rate sensitivities

We estimate firm-specific sensitivities to **daily changes in the 2Y EUR OIS** rate:

$$
r_{i,t} = a_i + b_i \Delta y_t + \varepsilon_{i,t}
\quad\Rightarrow\quad
D^{emp}_i = -b_i
$$

### 6.1 Pull the daily 2Y OIS series (RIC `EUREON2Y=`)

We use the field that is available in your setup: `TR.MIDPRICE` (rate in **percent**).


In [ ]:
ld.open_session()

# 2Y EUR OIS / EONIA-style series
ric_rate = "EUREON2Y="
field_rate = "TR.MIDPRICE"

y_df = ld.get_history(
    universe=[ric_rate],
    fields=[field_rate],
    start="1999-01-01",
    end="2025-12-31",
    interval="daily"
)

ld.close_session()

# Build daily rate changes (in percentage points)
rate_df = y_df.copy().reset_index()
rate_df.columns = ["date", "y"]   # y in percent
rate_df["date"] = pd.to_datetime(rate_df["date"])
rate_df["y"] = pd.to_numeric(rate_df["y"], errors="coerce")
rate_df = rate_df.dropna(subset=["y"]).sort_values("date")
rate_df["dy"] = rate_df["y"].diff()

rates_daily = rate_df[["date","dy"]].dropna()

rates_daily.head()


In [ ]:
df_2yOIS = rates_daily.copy()
save_parquet(df_2yOIS, "rates_2yOIS_daily")

### 6.2 Pull daily stock prices and compute daily returns

We retrieve daily prices for all RICs, compute simple returns, then merge with `rates_daily`.


In [ ]:
# -----------------------------
# Helpers
# -----------------------------
def chunk_list(x, n=30):
    for i in range(0, len(x), n):
        yield x[i:i+n]

def make_date_blocks(start: str, end: str, mode: str = "5y"):
    """
    mode: "yearly" or "5y"
    returns list of (start, end) as YYYY-MM-DD strings (inclusive endpoints for API calls)
    """
    start_dt = pd.Timestamp(start)
    end_dt   = pd.Timestamp(end)

    blocks = []
    cur = start_dt

    if mode not in {"yearly", "5y"}:
        raise ValueError("mode must be 'yearly' or '5y'")

    step_years = 1 if mode == "yearly" else 5

    while cur <= end_dt:
        nxt = cur + pd.DateOffset(years=step_years) - pd.DateOffset(days=1)
        blk_end = min(nxt, end_dt)
        blocks.append((cur.strftime("%Y-%m-%d"), blk_end.strftime("%Y-%m-%d")))
        cur = blk_end + pd.DateOffset(days=1)

    return blocks

def _safe_get_history(batch, field, start, end, interval="daily",
                      max_retries=5, base_sleep=0.8):
    """
    Robust wrapper around ld.get_history with retry/backoff.
    Returns df or None.
    """
    last_err = None
    for r in range(max_retries):
        try:
            df = ld.get_history(
                universe=batch,
                fields=[field],
                start=start,
                end=end,
                interval=interval
            )
            if df is not None and not df.empty:
                return df
        except Exception as e:
            last_err = e

        # exponential backoff + jitter
        sleep_s = base_sleep * (2 ** r) + random.random() * 0.5
        time.sleep(sleep_s)

    if last_err is not None:
        print(f"[WARN] Batch failed after retries. ({start}..{end}) Example RIC: {batch[0]} | {last_err}")
    else:
        print(f"[WARN] Batch returned empty after retries. ({start}..{end}) Example RIC: {batch[0]}")
    return None

def _ensure_wide_with_date(df, batch):
    """
    Tries to convert the returned object into a DataFrame with:
    columns: ["date"] + batch_rics_present
    """
    dfw = df.copy()

    # ensure date column
    if isinstance(dfw.index, pd.DatetimeIndex):
        dfw = dfw.reset_index()
        # name could be None or something else
        if "date" not in [c.lower() for c in dfw.columns]:
            dfw = dfw.rename(columns={dfw.columns[0]: "date"})
        else:
            # if there is already a date-like column name, unify it to "date"
            date_col = next((c for c in dfw.columns if str(c).lower() == "date"), dfw.columns[0])
            if date_col != "date":
                dfw = dfw.rename(columns={date_col: "date"})
    else:
        dfw = dfw.reset_index()
        date_col = next((c for c in dfw.columns if "date" in str(c).lower()), None)
        if date_col is None:
            date_col = dfw.columns[0]
        if date_col != "date":
            dfw = dfw.rename(columns={date_col: "date"})

    dfw["date"] = pd.to_datetime(dfw["date"])

    # keep only date + columns matching rics in batch (intersection)
    keep_cols = ["date"] + [c for c in dfw.columns if c in set(batch)]
    dfw = dfw[keep_cols].copy()

    return dfw

def _wide_to_long(dfw, value_name="value"):
    """
    Converts wide (date + RIC columns) to long (date, RIC, value).
    """
    long = dfw.melt(id_vars=["date"], var_name="RIC", value_name=value_name)
    long["value"] = pd.to_numeric(long["value"], errors="coerce")
    long = long.dropna(subset=["value"])
    return long

def infer_totalreturn_is_level(rets_long: pd.DataFrame, sample_rics=8):
    """
    Heuristic:
    - If typical absolute values are small (e.g., around 0.0xx), it's likely already returns.
    - If values are large and positive (e.g., 80, 100, 200) and mostly > 0, it's likely an index/level.
    We test on a sample of RICs.
    """
    if rets_long.empty:
        return False

    sample = (rets_long
              .sort_values(["RIC", "date"])
              .groupby("RIC", as_index=False)
              .head(50))

    # reduce to few RICs
    rics = sample["RIC"].dropna().unique()[:sample_rics]
    sample = sample[sample["RIC"].isin(rics)]

    vals = sample["value"].dropna().values
    if vals.size == 0:
        return False

    p50 = np.nanmedian(vals)
    p90 = np.nanpercentile(vals, 90)

    # Returns typically in [-0.2, 0.2] daily; levels typically much larger.
    # This is a heuristic; you can tighten/loosen thresholds.
    looks_like_level = (p90 > 5) and (p50 > 1)  # e.g., 100-ish index
    return bool(looks_like_level)

# -----------------------------
# Main pull: TR.TotalReturn in time blocks + batches
# -----------------------------
def get_total_return_daily(
    rics,
    start="1999-01-01",
    end=None,
    field="TR.TotalReturn",
    batch_size=30,
    block_mode="5y",              # "yearly" or "5y"
    checkpoint_dir=None,          # e.g. Path(".../intermediate/total_return_blocks")
    max_retries=5
):
    """
    Pull TR.TotalReturn in (time blocks x RIC batches), with retries and optional checkpoints.
    Returns long: date, RIC, value (raw from API)
    """
    # default end = today
    if end is None:
        end = pd.Timestamp.today().strftime("%Y-%m-%d")

    # clean RICs
    rics = [r for r in pd.Series(rics).dropna().astype(str).unique().tolist() if r.strip() != ""]
    blocks = make_date_blocks(start, end, mode=block_mode)

    if checkpoint_dir is not None:
        checkpoint_dir = Path(checkpoint_dir)
        checkpoint_dir.mkdir(parents=True, exist_ok=True)

    out = []

    for blk_ix, (blk_start, blk_end) in enumerate(blocks, start=1):
        for b_ix, batch in enumerate(chunk_list(rics, n=batch_size), start=1):

            ck_path = None
            if checkpoint_dir is not None:
                ck_path = checkpoint_dir / f"blk_{blk_ix:03d}_{blk_start}_{blk_end}__batch_{b_ix:03d}.parquet"
                if ck_path.exists():
                    out.append(pd.read_parquet(ck_path))
                    continue

            df = _safe_get_history(
                batch=batch,
                field=field,
                start=blk_start,
                end=blk_end,
                interval="daily",
                max_retries=max_retries
            )
            if df is None or df.empty:
                continue

            dfw = _ensure_wide_with_date(df, batch)
            if dfw.shape[1] <= 1:
                continue

            df_long = _wide_to_long(dfw, value_name="value")

            if ck_path is not None:
                df_long.to_parquet(ck_path, index=False)

            out.append(df_long)

        print(f"[INFO] Finished block {blk_ix}/{len(blocks)}: {blk_start}..{blk_end}")

    if not out:
        return pd.DataFrame(columns=["date", "RIC", "value"])

    res = pd.concat(out, ignore_index=True)
    res["date"] = pd.to_datetime(res["date"])
    return res

def to_daily_returns_from_totalreturn(raw_long: pd.DataFrame):
    """
    Converts raw TR.TotalReturn output to daily returns if it looks like a level/index.
    If it already looks like returns, keeps as-is.

    Output: date, RIC, ret
    """
    if raw_long.empty:
        return pd.DataFrame(columns=["date", "RIC", "ret"])

    raw_long = raw_long.sort_values(["RIC", "date"]).copy()

    is_level = infer_totalreturn_is_level(raw_long)
    if is_level:
        # Treat as level/index -> convert to returns
        raw_long["ret"] = raw_long.groupby("RIC")["value"].pct_change()
        out = raw_long.dropna(subset=["ret"])[["date", "RIC", "ret"]].copy()
        print("[INFO] TR.TotalReturn looked like a LEVEL/INDEX. Converted via pct_change().")
        return out
    else:
        # Treat as already a return series
        out = raw_long.rename(columns={"value": "ret"})[["date", "RIC", "ret"]].copy()
        print("[INFO] TR.TotalReturn looked like RETURNS already. Kept as-is.")
        return out

# -----------------------------
# Example usage
# -----------------------------
# Optional: choose a checkpoint folder (highly recommended for big pulls)
# checkpoint_dir = DATA_DIR / "tr_totalreturn_checkpoints"
checkpoint_dir = None

ld.open_session()

raw_tr = get_total_return_daily(
    rics=df_basic["RIC"].tolist(),
    start="1999-01-01",
    end=None,                  # <-- (2) automatically today
    field="TR.TotalReturn",
    batch_size=30,
    block_mode="5y",           # <-- (3) "yearly" or "5y"
    checkpoint_dir=checkpoint_dir,
    max_retries=5
)

ld.close_session()

rets_daily = to_daily_returns_from_totalreturn(raw_tr)
rets_daily = rets_daily.sort_values(["RIC", "date"]).reset_index(drop=True)

rets_daily.head()

/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-pac

[INFO] Finished block 1/6: 1999-01-01..2003-12-31


/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-pac

[INFO] Finished block 2/6: 2004-01-01..2008-12-31


/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-pac

...

/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-pac

[INFO] Finished block 5/6: 2019-01-01..2023-12-31


/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/lseg/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-pac

[INFO] Finished block 6/6: 2024-01-01..2026-01-22
[INFO] TR.TotalReturn looked like RETURNS already. Kept as-is.


,date,RIC,ret
0,2015-10-07,1COVG.DE^L25,-0.377358
1,2015-10-08,1COVG.DE^L25,-1.590909
2,2015-10-09,1COVG.DE^L25,0.846805
3,2015-10-12,1COVG.DE^L25,-0.763359
4,2015-10-13,1COVG.DE^L25,-1.153846


In [1]:
rets_daily.info()
rets_daily.head()
rets_daily.tail()

# Plausibilitätschecks
rets_daily.groupby("RIC")["date"].agg(["min", "max"]).head()
rets_daily["ret"].describe()

NameError: name 'rets_daily' is not defined

In [2]:
save_parquet(rets_daily, "returns_daily")

NameError: name 'save_parquet' is not defined

In [3]:
# erste 30 RICs auswählen (stabil, alphabetisch)
first_30_rics = (
    rets_daily["RIC"]
    .drop_duplicates()
    .sort_values()
    .head(30)
    .tolist()
)

# pivot: date = rows, RIC = columns
preview_wide = (
    rets_daily
    .query("RIC in @first_30_rics")
    .pivot(index="date", columns="RIC", values="ret")
    .sort_index()
)

preview_wide.head()

NameError: name 'rets_daily' is not defined

### 6.3 Estimate firm-level rate betas and empirical duration

In [ ]:
def estimate_rate_beta_duration(rets_daily: pd.DataFrame, rates_daily: pd.DataFrame, min_obs: int = 200) -> pd.DataFrame:
    """Estimate r_{i,t} = a_i + b_i dy_t; return b_i and D_emp=-b_i."""
    df = rets_daily.merge(rates_daily, on="date", how="inner").dropna(subset=["ret","dy"]).copy()

    out = []
    for ric, g in df.groupby("RIC"):
        if len(g) < min_obs:
            continue
        x = g["dy"].to_numpy(float)   # dy in percentage points
        y = g["ret"].to_numpy(float)

        X = np.column_stack([np.ones_like(x), x])
        alpha, b = np.linalg.lstsq(X, y, rcond=None)[0]
        out.append({"RIC": ric, "beta_rate_2yOIS": b, "D_emp_2yOIS": -b, "n_obs": len(g)})

    return pd.DataFrame(out)

dur_emp = estimate_rate_beta_duration(rets_daily, rates_daily, min_obs=200)
df_basic = df_basic.merge(dur_emp, on="RIC", how="left")

print(df_basic[["RIC","D_emp_2yOIS","beta_rate_2yOIS","n_obs"]].head())
print(df_basic["D_emp_2yOIS"].describe())
print(df_basic[["Duration_r125","D_emp_2yOIS"]].corr(method="spearman"))
print("Share with D_emp:", df_basic["D_emp_2yOIS"].notna().mean())


## 7. Net-Payout-Based Duration

## 8. Final results table (rounded)

We include:
- Firm descriptors
- CAPM inputs
- CFPS forecasts
- All duration measures (rounded to 2 decimals)


In [ ]:
cols = [
    "RIC","CompanyName","CountryHQ", "Price_2025_12_31",
    "ROE_FY0","BETA_5Y","COE_CAPM",
    "CFPSMean_FY25","CFPSMean_FY26","CFPSMean_FY27","CFPSMean_FY28","CFPSMean_FY29",
    "Duration_r125","Duration_CAPM","Duration_undiscounted" #,"D_emp_2yOIS"
]

df_results = df_basic[cols].copy()

# percentages for presentation
df_results["ROE_FY0"] = df_results["ROE_FY0"] * 100
df_results["COE_CAPM"] = df_results["COE_CAPM"] * 100

df_results = df_results.rename(columns={
    "Price_2025_12_31": "Price (Dec 31, 2025)",
    "ROE_FY0": "ROE (%)",
    "BETA_5Y": "Beta (5Y)",
    "COE_CAPM": "Cost of Equity CAPM (%)",
    "CFPSMean_FY25": "CFPS FY25",
    "CFPSMean_FY26": "CFPS FY26",
    "CFPSMean_FY27": "CFPS FY27",
    "CFPSMean_FY28": "CFPS FY28",
    "CFPSMean_FY29": "CFPS FY29",
    "Duration_r125": "Duration (r = 12.5%)",
    "Duration_CAPM": "Duration (CAPM r)",
    "Duration_undiscounted": "Duration (undiscounted)",
    #"D_emp_2yOIS": "Empirical Duration (2Y OIS)"
})

df_results.to_excel(TABLE_DIR / "final_results_duration.xlsx", index=False)

df_results_rounded = df_results.copy()

# round most numeric columns
df_results_rounded = df_results_rounded.round({
    "Price (Dec 31, 2025)": 2,
    "ROE (%)": 2,
    "Beta (5Y)": 2,
    "Cost of Equity CAPM (%)": 2,
    "CFPS FY25": 2, "CFPS FY26": 2, "CFPS FY27": 2, "CFPS FY28": 2, "CFPS FY29": 2,
})

# round all duration columns automatically
duration_cols = [c for c in df_results_rounded.columns if "Duration" in c]
df_results_rounded[duration_cols] = df_results_rounded[duration_cols].round(2)

df_results_rounded.head(20)
